# exp01 - PharmaSales daily: perbandingan di bawah protokol tunggal

Notebook ini **menggantikan** perbandingan lintas-notebook pada revisi sebelumnya
(`our_study_pharma_daily.ipynb` vs `rathipriya_pharma_daily.ipynb`), yang oleh kedua
reviewer IJIES dinilai tidak sah karena setiap model dijalankan dengan preprocessing,
pembagian data, dan protokol tuning yang berbeda.

## Kontrak eksperimen

Semua angka di notebook ini dihasilkan di bawah satu kontrak yang dikodekan di
`src/experiments/protocol.py` (satu-satunya sumber kebenaran; notebook lain memakai
modul yang sama sehingga protokolnya tidak mungkin menyimpang):

| Aspek | Ketentuan |
|---|---|
| Seed | `SEED = 42`, dipasang ke `random`, NumPy, `PYTHONHASHSEED`; setiap estimator stokastik menerima `random_state=SEED` melalui satu konstruktor `make_xgb()` |
| Split | Kronologis 70 / 15 / 15 (train / validation / test), tanpa pengacakan, **identik untuk semua model dan semua set fitur** |
| Tuning | Hyperparameter dipilih **hanya** dari RMSE validation. Test split disentuh **satu kali** oleh model final |
| Refit | Model final di-refit pada train+val dengan konfigurasi terbaik dari validation |
| Fitur | Set fitur adalah **faktor eksperimen**: `A_lag1` (protokol referensi) dan `B_rich` (lag terpilih + rolling mean) - setiap model dijalankan pada keduanya |
| Seleksi lag | argmax PACF dihitung **hanya pada blok training**; aturan alternatif diuji sebagai ablasi tersendiri |
| Penskalaan | Scaler di-fit ulang pada blok training aktif saja, tidak pernah pada test |

## Perbedaan terhadap pipeline lama (dan alasannya)

1. **Baseline dan model usulan memakai fitur serta split yang sama.** Sebelumnya baris
   "Reference" pada Tabel 2 hanya diberi `lag_1` dengan split 70/15/15, sedangkan model
   usulan memakai 2-16 fitur dengan split 60/20/20. Selisih yang dilaporkan karena itu
   mencampur efek model dengan efek preprocessing.
2. **Tidak ada grid search pada test set.** Cabang "Our Preprocessing" pada notebook
   baseline memilih hyperparameter dengan menilai test split (80/20 tanpa validation).
3. **PACF dihitung pada blok training saja.** Notebook lama menghitung ACF/PACF pada
   seluruh deret termasuk test - kebocoran halus pada tahap desain fitur.
4. **Grid bandwidth kernel diperlebar.** Pada grid lama (maksimum sigma = 5) optimum
   validation jatuh persis di batas atas untuk sebagian besar kategori, artinya grid
   memotong ruang pencarian dan secara sistematis merugikan GRNN/P_NN.
5. **Baseline naif ditambahkan.** Klaim keunggulan tanpa pembanding naif tidak dapat
   dinilai; Naive dan Seasonal Naive (s=7) kini dilaporkan di setiap tabel.
6. **Uji Diebold-Mariano dilaporkan** agar selisih RMSE dapat dinilai signifikansinya,
   bukan sekadar dibandingkan angkanya.

In [2]:
import sys, os, warnings
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.experiments import protocol as P

P.set_global_seed()

GRANULARITY      = "daily"
DATA_PATH        = "../data/raw/pharma-sales/salesdaily.csv"
SEASONAL_PERIOD  = 7
EXPERIMENT       = "exp01_pharma_daily"
CATEGORIES       = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

data = pd.read_csv(DATA_PATH)
print("Lingkungan:", P.environment_stamp())
print("Baris data mentah:", len(data))
data.head()

Lingkungan: {'python': '3.10.11', 'platform': 'Windows-10-10.0.22621-SP0', 'numpy': '2.2.6', 'pandas': '2.3.3', 'seed': '42', 'sklearn': '1.7.2', 'xgboost': '3.2.0', 'statsmodels': '0.14.6'}
Baris data mentah: 2106


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,Month,Hour,Weekday Name
0,1/2/2014,0.0,3.67,3.4,32.40,7.0,0.0,0.0,2.0,2014,1,248,Thursday
1,1/3/2014,8.0,4.00,4.4,50.60,16.0,0.0,20.0,4.0,2014,1,276,Friday
2,1/4/2014,2.0,1.00,6.5,61.85,10.0,0.0,9.0,1.0,2014,1,276,Saturday
3,1/5/2014,4.0,3.00,7.0,41.10,8.0,0.0,3.0,0.0,2014,1,276,Sunday
4,1/6/2014,5.0,1.00,4.5,21.70,16.0,2.0,6.0,2.0,2014,1,276,Monday


## 1. Stabilitas aturan pemilihan lag

Nilai *k* (jumlah lag) menentukan seluruh set fitur `B_rich`. Notebook lama memakai
**dua aturan berbeda** untuk dua pipeline yang seharusnya dibandingkan: `our_study`
memakai argmax PACF, `rathipriya` (cabang "Our Preprocessing") memakai argmax ACF, dan
keduanya menghitung statistiknya pada deret penuh termasuk test.

Tabel di bawah menunjukkan bahwa *k* berubah drastis hanya karena pilihan tersebut.
Ini bukan detail teknis: bila *k* tidak stabil, kontribusi tahap rekayasa fitur pada
metode usulan juga tidak stabil, dan itu harus dilaporkan.

In [3]:
lag_table = []
for category in CATEGORIES:
    y = data[category].to_numpy(dtype=float)
    row = {"Category": category}
    for rule in ["pacf_train", "pacf_full", "acf_train", "acf_full", "pacf_significant"]:
        row[rule] = P.select_lag(y, rule=rule)
    lag_table.append(row)

lag_table = pd.DataFrame(lag_table)
print("k (jumlah lag) menurut aturan seleksi -- *_full memakai data test (bocor)")
lag_table

k (jumlah lag) menurut aturan seleksi -- *_full memakai data test (bocor)


,Category,pacf_train,pacf_full,acf_train,acf_full,pacf_significant
0,M01AB,1,6,14,14,15
1,M01AE,21,1,21,14,22
2,N02BA,1,1,14,7,18
3,N02BE,1,1,7,7,21
4,N05B,1,1,1,1,25
5,N05C,7,4,7,4,7
6,R03,1,1,1,15,22
7,R06,1,1,2,3,21


## 2. Desain split

Split dicetak secara eksplisit (tanggal awal/akhir setiap blok, ukuran setiap blok)
karena reviewer secara khusus meminta tanggal train/test yang hilang dari naskah.
Perhatikan bahwa `A_lag1` dan `B_rich` berbagi **baris dan tanggal yang persis sama**:
kedua set fitur dibangun dari kerangka yang sama dengan *k* yang sama, sehingga
satu-satunya yang berbeda adalah kolom fiturnya.

In [4]:
datasets = {}
split_rows = []
for category in CATEGORIES:
    for feature_set in P.FEATURE_SETS:
        d = P.build_pharma_dataset(data, category, feature_set,
                                   seasonal_period=SEASONAL_PERIOD,
                                   lag_rule="pacf_train")
        datasets[(category, feature_set)] = d
        split_rows.append({**d.describe(), "features": ", ".join(d.feature_names[:4]) +
                           (" ..." if len(d.feature_names) > 4 else "")})

split_table = pd.DataFrame(split_rows)
split_table.to_csv(f"../results/{EXPERIMENT}_splits.csv", index=False)
split_table

,category,feature_set,lag_rule,n_features,n_lags,n_train,n_val,n_test,train_start,train_end,val_start,val_end,test_start,test_end,features
0,M01AB,A_lag1,pacf_train,1,1,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,lag_1
1,M01AB,B_rich,pacf_train,1,1,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,lag_1
2,M01AE,A_lag1,pacf_train,1,21,1459,312,314,2014-01-23,2018-01-20,2018-01-21,2018-11-28,2018-11-29,2019-10-08,lag_1
3,M01AE,B_rich,pacf_train,22,21,1459,312,314,2014-01-23,2018-01-20,2018-01-21,2018-11-28,2018-11-29,2019-10-08,"lag_1, lag_2, lag_3, lag_4 ..."
4,N02BA,A_lag1,pacf_train,1,1,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,lag_1
5,N02BA,B_rich,pacf_train,1,1,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,lag_1
6,N02BE,A_lag1,pacf_train,1,1,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,lag_1
7,N02BE,B_rich,pacf_train,1,1,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,lag_1
8,N05B,A_lag1,pacf_train,1,1,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,lag_1
9,N05B,B_rich,pacf_train,1,1,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,lag_1


## 3. Menjalankan seluruh model

Setiap model dieksekusi lewat `P.run_model`, yang menegakkan kontrak tuning:
grid dievaluasi pada validation, konfigurasi terbaik di-refit pada train+val, lalu
test diprediksi tepat satu kali. Tidak ada jalur kode di mana test dapat memengaruhi
pemilihan hyperparameter.

`XGBoost`, `LR+XGB (rata-rata)` dan `LR-XGB (residual)` memakai grid yang sama
(`GRID_XGB_PHARMA`, 12 konfigurasi) agar anggaran tuningnya setara.

In [5]:
MODELS = [
    # (nama, fungsi fit_predict, grid, jenis scaler)
    ("LR",                  P.fp_linear_regression, None,               None),
    ("GRNN",                P.fp_grnn,              P.GRID_GRNN,        "standard"),
    ("P_NN",                P.fp_pnn,               P.GRID_PNN,         "standard"),
    ("RBFNN",               P.fp_rbfnn,             P.GRID_RBFNN,       "standard"),
    ("XGBoost",             P.fp_xgboost,           P.GRID_XGB_PHARMA,  None),
    ("LR+XGB (average)",    P.fp_lr_xgb_average,    P.GRID_XGB_PHARMA,  None),
    ("LR-XGB (residual)",   P.fp_lr_xgb_residual,   P.GRID_XGB_PHARMA,  None),
]

rows = []
for category in CATEGORIES:
    for feature_set in P.FEATURE_SETS:
        d = datasets[(category, feature_set)]
        rows += P.naive_rows(d)
        for name, fn, grid, scaler in MODELS:
            rows.append(P.run_model(name, fn, d, grid, scaler_kind=scaler))
    print(f"selesai: {category}", flush=True)

# ARIMA(5,1,0) bersifat univariat: tidak bergantung pada set fitur, dijalankan sekali
# per kategori dan dilaporkan pada kedua set fitur dengan penanda eksplisit.
for category in CATEGORIES:
    d = datasets[(category, P.FEATURE_SET_A)]
    try:
        rows.append(P.run_model("ARIMA(5,1,0)", P.fp_arima, d,
                                {"order": [(5, 1, 0)]},
                                extra={"note": "univariat; tidak memakai fitur"}))
    except Exception as exc:
        print(f"ARIMA gagal untuk {category}: {exc}")

results = P.save_results(rows, EXPERIMENT)
print(f"{len(results)} baris hasil ditulis ke ../results/{EXPERIMENT}.csv")
results.head()

selesai: M01AB
selesai: M01AE
selesai: N02BA
selesai: N02BE
selesai: N05B
selesai: N05C
selesai: R03
selesai: R06
152 baris hasil ditulis ke ../results/exp01_pharma_daily.csv


,model,category,feature_set,lag_rule,n_features,n_lags,n_train,n_val,n_test,train_start,train_end,val_start,val_end,test_start,test_end,params,n_grid,scaler,seed,val_MSE,val_RMSE,val_MAE,val_R2,val_nRMSE,val_SMAPE,val_RMSPE,test_MSE,test_RMSE,test_MAE,test_R2,test_nRMSE,test_SMAPE,test_RMSPE,runtime_s,note
0,Naive,M01AB,A_lag1,pacf_train,1,1,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,{},0,none,42,15.112756,3.887513,2.996063,-1.088908,0.787629,64.504381,1.556892,17.410850,4.172631,3.291483,-1.077907,0.784236,66.446574,2.203789,0.000,NaN
1,SeasonalNaive(s=7),M01AB,A_lag1,pacf_train,1,1,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,{},0,none,42,13.791767,3.713727,2.996413,-0.906319,0.752419,64.365336,1.766191,16.361037,4.044878,3.161136,-0.952617,0.760225,64.380072,2.596577,0.000,NaN
2,LR,M01AB,A_lag1,pacf_train,1,1,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,{},1,none,42,7.360533,2.713030,2.175528,-0.017384,0.549673,47.367245,1.372989,8.566003,2.926774,2.263650,-0.022314,0.550080,46.915711,1.864104,0.493,NaN
3,GRNN,M01AB,A_lag1,pacf_train,1,1,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,"{""sigma"": 50.0}",16,standard,42,7.238455,2.690438,2.163435,-0.000510,0.545096,47.218316,1.379301,8.491273,2.913979,2.247760,-0.013395,0.547675,46.700385,1.854202,0.669,NaN
4,P_NN,M01AB,A_lag1,pacf_train,1,1,1473,315,317,2014-01-03,2018-01-14,2018-01-15,2018-11-25,2018-11-26,2019-10-08,"{""sigma"": 0.2}",16,standard,42,8.158243,2.856264,2.235103,-0.127645,0.578693,49.454904,1.099066,10.129578,3.182700,2.456986,-0.208920,0.598181,51.694293,1.709154,0.740,NaN


## 4. Tabel utama - RMSE test per kategori x set fitur

Ini adalah tabel yang menggantikan Tabel 2 pada naskah. Semua sel dihasilkan dari
satu proses, satu split, satu protokol tuning, dan satu seed.

In [6]:
pivot = results.pivot_table(index=["category", "feature_set"], columns="model",
                            values="test_RMSE")
order = ["Naive", f"SeasonalNaive(s={SEASONAL_PERIOD})", "ARIMA(5,1,0)", "LR",
         "GRNN", "P_NN", "RBFNN", "XGBoost", "LR+XGB (average)", "LR-XGB (residual)"]
pivot = pivot[[c for c in order if c in pivot.columns]]
display(pivot.round(4))

winner = pivot.idxmin(axis=1).rename("model terbaik (RMSE test)")
print("\nModel terbaik per (kategori, set fitur):")
print(winner.to_string())
print("\nBerapa kali metode usulan menang:",
      int((winner == "LR-XGB (residual)").sum()), "dari", len(winner))

model                   Naive  SeasonalNaive(s=7)  ARIMA(5,1,0)       LR     GRNN     P_NN    RBFNN  XGBoost  LR+XGB (average)  LR-XGB (residual)
category feature_set                                                                                                                             
M01AB    A_lag1        4.1726              4.0449        2.9932   2.9268   2.9140   3.1827   2.9229   3.0326            2.9724             3.0338
         B_rich        4.1726              4.0449           NaN   2.9268   2.9140   3.1827   2.9229   3.0326            2.9724             3.0338
M01AE    A_lag1        2.8509              2.8647        2.4341   2.3220   2.3544   2.5472   2.3035   2.3064            2.3109             2.3027
         B_rich        2.8509              2.8647           NaN   2.2025   2.2942   2.3599   2.2625   2.2365            2.2110             2.1891
N02BA    A_lag1        2.7857              2.6936        2.0357   2.1554   2.1503   2.2020   2.1551   2.1633            2.1545             2.1631
         B_rich        2.7857              2.6936           NaN   2.1554   2.1503   2.2020   2.1551   2.1633            2.1545             2.1631
N02BE    A_lag1       16.3589             16.3242       17.3057  14.3395  14.3567  14.8283  14.2656  14.2650           14.2309            14.2362
         B_rich       16.3589             16.3242           NaN  14.3395  14.3567  14.8283  14.2656  14.2650           14.2309            14.2362
N05B     A_lag1        5.8983              5.7878        4.3257   4.4338   4.3883   5.9764   4.4381   4.4584            4.4741             4.4602
         B_rich        5.8983              5.7878           NaN   4.4338   4.3883   5.9764   4.4381   4.4584            4.4741             4.4602
N05C     A_lag1        1.6621              1.6275        1.1353   1.1452   1.1433   1.2709   1.1442   1.1448            1.1446             1.1449
         B_rich        1.6621              1.6275           NaN   1.1471   1.1455   1.2709   1.1461   1.1599            1.1558             1.1946
R03      A_lag1       10.8952             10.8831        8.5551   8.2880   8.3497  10.2576   8.3298   8.3909            8.3223             8.4016
         B_rich       10.8952             10.8831           NaN   8.2880   8.3497  10.2576   8.3298   8.3909            8.3223             8.4016
R06      A_lag1        3.0094              3.1969        3.2667   2.5460   2.5546   3.1865   2.5512   2.5482            2.5434             2.5461
         B_rich        3.0094              3.1969           NaN   2.5460   2.5546   3.1865   2.5512   2.5482            2.5434             2.5461


Model terbaik per (kategori, set fitur):
category  feature_set
M01AB     A_lag1                      GRNN
          B_rich                      GRNN
M01AE     A_lag1         LR-XGB (residual)
          B_rich         LR-XGB (residual)
N02BA     A_lag1              ARIMA(5,1,0)
          B_rich                      GRNN
N02BE     A_lag1          LR+XGB (average)
          B_rich          LR+XGB (average)
N05B      A_lag1              ARIMA(5,1,0)
          B_rich                      GRNN
N05C      A_lag1              ARIMA(5,1,0)
          B_rich                      GRNN
R03       A_lag1                        LR
          B_rich                        LR
R06       A_lag1          LR+XGB (average)
          B_rich          LR+XGB (average)

Berapa kali metode usulan menang: 2 dari 16


## 5. Apakah selisihnya signifikan? (Diebold-Mariano)

Selisih RMSE beberapa persen pada n_test kecil tidak otomatis berarti model lebih baik.
Uji DM di bawah membandingkan metode usulan dengan (a) baseline terbaik non-hibrida
pada kondisi yang sama dan (b) Seasonal Naive. Nilai DM negatif berarti metode usulan
lebih akurat; `p_value` di atas 0.05 berarti selisihnya tidak dapat dibedakan dari nol.

In [7]:
by_key = {(r["category"], r["feature_set"], r["model"]): r for r in rows}
dm_rows = []
for category in CATEGORIES:
    for feature_set in P.FEATURE_SETS:
        prop = by_key.get((category, feature_set, "LR-XGB (residual)"))
        if prop is None:
            continue
        d = datasets[(category, feature_set)]
        competitors = [m for m in ["LR", "GRNN", "P_NN", "RBFNN", "XGBoost",
                                   f"SeasonalNaive(s={SEASONAL_PERIOD})"]
                       if (category, feature_set, m) in by_key]
        best = min(competitors,
                   key=lambda m: by_key[(category, feature_set, m)]["test_RMSE"])
        for other in {best, f"SeasonalNaive(s={SEASONAL_PERIOD})"}:
            ref = by_key[(category, feature_set, other)]
            test = P.diebold_mariano(d.y_test, prop["_test_pred"], ref["_test_pred"])
            dm_rows.append({
                "category": category, "feature_set": feature_set,
                "pembanding": other,
                "RMSE usulan": round(prop["test_RMSE"], 4),
                "RMSE pembanding": round(ref["test_RMSE"], 4),
                "DM": round(test["DM"], 3) if test["DM"] == test["DM"] else None,
                "p_value": round(test["p_value"], 4) if test["p_value"] == test["p_value"] else None,
                "signifikan (a=0.05)": (test["p_value"] < 0.05) if test["p_value"] == test["p_value"] else None,
            })

dm_table = pd.DataFrame(dm_rows).sort_values(["category", "feature_set", "pembanding"])
dm_table.to_csv(f"../results/{EXPERIMENT}_dm_test.csv", index=False)
dm_table

,category,feature_set,pembanding,RMSE usulan,RMSE pembanding,DM,p_value,signifikan (a=0.05)
1,M01AB,A_lag1,GRNN,3.0338,2.9140,3.480,0.0006,True
0,M01AB,A_lag1,SeasonalNaive(s=7),3.0338,4.0449,-5.834,0.0000,True
3,M01AB,B_rich,GRNN,3.0338,2.9140,3.480,0.0006,True
2,M01AB,B_rich,SeasonalNaive(s=7),3.0338,4.0449,-5.834,0.0000,True
5,M01AE,A_lag1,RBFNN,2.3027,2.3035,-0.054,0.9568,False
4,M01AE,A_lag1,SeasonalNaive(s=7),2.3027,2.8647,-4.515,0.0000,True
7,M01AE,B_rich,LR,2.1891,2.2025,-0.741,0.4593,False
6,M01AE,B_rich,SeasonalNaive(s=7),2.1891,2.8647,-5.868,0.0000,True
9,N02BA,A_lag1,GRNN,2.1631,2.1503,1.183,0.2379,False
8,N02BA,A_lag1,SeasonalNaive(s=7),2.1631,2.6936,-3.996,0.0001,True


## 6. Ablasi aturan seleksi lag

Faktor ketiga: apakah kesimpulan berubah bila *k* dipilih dengan aturan lain?
Bagian ini menjalankan ulang metode usulan dan LR di bawah lima aturan seleksi lag.
Bila peringkat model berubah-ubah antar aturan, klaim keunggulan harus dilemahkan
secara eksplisit di naskah.

In [8]:
ablation_rows = []
for rule in ["pacf_train", "pacf_full", "acf_train", "acf_full", "pacf_significant"]:
    for category in CATEGORIES:
        d = P.build_pharma_dataset(data, category, P.FEATURE_SET_B,
                                   seasonal_period=SEASONAL_PERIOD, lag_rule=rule)
        ablation_rows.append(P.run_model("LR", P.fp_linear_regression, d))
        ablation_rows.append(P.run_model("LR-XGB (residual)", P.fp_lr_xgb_residual,
                                         d, P.GRID_XGB_PHARMA))
    print("selesai aturan:", rule, flush=True)

ablation = P.save_results(ablation_rows, f"{EXPERIMENT}_lag_ablation")
ablation.pivot_table(index=["category", "lag_rule"], columns="model",
                     values="test_RMSE").round(4)

selesai aturan: pacf_train
selesai aturan: pacf_full
selesai aturan: acf_train
selesai aturan: acf_full
selesai aturan: pacf_significant


model                           LR  LR-XGB (residual)
category lag_rule                                    
M01AB    acf_full           2.8983             2.9104
         acf_train          2.8983             2.9104
         pacf_full          2.9151             2.9180
         pacf_significant   2.9090             2.9100
         pacf_train         2.9268             3.0338
M01AE    acf_full           2.2037             2.1878
         acf_train          2.2025             2.1891
         pacf_full          2.3144             2.3003
         pacf_significant   2.2015             2.1865
         pacf_train         2.2025             2.1891
N02BA    acf_full           2.0727             2.0849
         acf_train          2.0234             2.0295
         pacf_full          2.1554             2.1631
         pacf_significant   2.0108             2.0294
         pacf_train         2.1554             2.1631
N02BE    acf_full          12.7470            12.6990
         acf_train         12.7470            12.6990
         pacf_full         14.3395            14.2362
         pacf_significant  12.7255            12.9632
         pacf_train        14.3395            14.2362
N05B     acf_full           4.4338             4.4602
         acf_train          4.4338             4.4602
         pacf_full          4.4338             4.4602
         pacf_significant   4.3682             4.3460
         pacf_train         4.4338             4.4602
N05C     acf_full           1.1436             1.1422
         acf_train          1.1471             1.1946
         pacf_full          1.1436             1.1422
         pacf_significant   1.1471             1.1946
         pacf_train         1.1471             1.1946
R03      acf_full           7.8318             7.8419
         acf_train          8.2880             8.4016
         pacf_full          8.2880             8.4016
         pacf_significant   7.8675             8.0927
         pacf_train         8.2880             8.4016
R06      acf_full           2.3733             2.4062
         acf_train          2.4511             2.4810
         pacf_full          2.5460             2.5461
         pacf_significant   2.3010             2.3217
         pacf_train         2.5460             2.5461

## 7. Pemeriksaan determinisme

Klaim reproduktifitas harus diuji, bukan dinyatakan. Sel di bawah menjalankan ulang
metode usulan untuk seluruh kategori pada proses yang sama dan membandingkan prediksi
bit-per-bit dengan hasil pertama.

In [9]:
P.set_global_seed()
identical = True
for category in CATEGORIES:
    d = datasets[(category, P.FEATURE_SET_B)]
    again = P.run_model("LR-XGB (residual)", P.fp_lr_xgb_residual, d, P.GRID_XGB_PHARMA)
    first = by_key[(category, P.FEATURE_SET_B, "LR-XGB (residual)")]
    same = np.array_equal(again["_test_pred"], first["_test_pred"])
    identical &= same
    print(f"{category:6s} prediksi identik: {same}  | RMSE {again['test_RMSE']:.6f} "
          f"vs {first['test_RMSE']:.6f}")

print("\nSEMUA IDENTIK:", identical)

M01AB  prediksi identik: True  | RMSE 3.033846 vs 3.033846
M01AE  prediksi identik: True  | RMSE 2.189096 vs 2.189096
N02BA  prediksi identik: True  | RMSE 2.163149 vs 2.163149
N02BE  prediksi identik: True  | RMSE 14.236194 vs 14.236194
N05B   prediksi identik: True  | RMSE 4.460172 vs 4.460172
N05C   prediksi identik: True  | RMSE 1.194611 vs 1.194611
R03    prediksi identik: True  | RMSE 8.401627 vs 8.401627
R06    prediksi identik: True  | RMSE 2.546119 vs 2.546119

SEMUA IDENTIK: True


## 8. Catatan untuk naskah

* Semua angka di notebook ini berada pada **skala asli** unit penjualan daily; tidak
  ada metrik yang dilaporkan pada skala transformasi tanpa penanda. Untuk setiap baris,
  `RMSE = sqrt(MSE)` secara eksak menurut konstruksi (`compute_metrics`), sehingga
  ketidakkonsistenan RMSE/MSE yang ditemukan reviewer tidak dapat terulang.
* `results/{EXPERIMENT}.csv` berisi satu baris per (kategori, set fitur, model) lengkap
  dengan hyperparameter terpilih, metrik validation, metrik test, ukuran split, tanggal
  split, dan seed - format machine-readable yang diminta reviewer.
* Bila kolom `n_features` untuk `A_lag1` dan `B_rich` bernilai sama pada suatu kategori,
  itu berarti argmax PACF pada blok training bernilai 1 sehingga `rolling_mean_1`
  identik dengan `lag_1` dan dibuang sebagai kolom duplikat. Kondisi ini dilaporkan
  apa adanya: untuk kategori tersebut pipeline "kaya" memang berdegenerasi menjadi
  pipeline referensi.